# Revisão do M6

Este notebook audita o M6 e define uma medida de planejamento inicial adequada à RQ3.

**Conclusão:** a nota LLM sobre assuntos de commit T1 não deve ser tratada sozinha como qualidade arquitetural. A proposta separa presença/escopo de artefatos de planejamento e conteúdo declarativo de planejamento, preservando ausência como informação distinta.

## Estrutura proposta

| Saída | Pergunta respondida | Relação com M7--M9 |
|---|---|---|
| **M6a - Presença e escopo estrutural** | Há artefatos de planejamento T1 e qual é seu alcance? | Complementa M7; não confunde ausência com baixa qualidade |
| **M6b - Conteúdo declarativo de planejamento** | Os assuntos de commits T1 declaram objetivos, arquitetura, tarefas ou decisões? | Substitui a nota LLM opaca, se reprocessada |
| **M6c - Concordância estrutural-textual** | Artefatos e mensagens contam a mesma história ou divergem? | Diagnóstico de evidência, não score composto |

M8 continua sendo desfecho de retrabalho e M9 deve associar cada componente M6 separadamente, sem criar um índice único.

## 1. Vínculo com a pergunta de pesquisa

O notebook confirma diretamente no texto atual do paper que M6 pertence à RQ3 e falha se essa vinculação mudar.

In [ ]:
from hashlib import sha256
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'paper_v8/latex_code/main.tex').is_file():
            return candidate
    raise FileNotFoundError('Could not locate project root')

PROJECT_ROOT = find_project_root(Path.cwd())
PAPER_PATH = PROJECT_ROOT / 'paper_v8/latex_code/main.tex'
M6_PATH = PROJECT_ROOT / 'paper_v8/data/m6_t1_planning_quality.csv'
SIGNALS_PATH = PROJECT_ROOT / 'paper_v4/advanced_metrics/outputs/team_level_signals.csv'
PLANNING_PATH = PROJECT_ROOT / 'data/analysis/planning_metrics.parquet'

paper_text = PAPER_PATH.read_text(encoding='utf-8')
rq_matches = dict(re.findall(r'\\item \\textbf\{(RQ\d)[^}]*:\}\s*(.*?)\n', paper_text))
m6_position = paper_text.index(r'\textbf{M6 --')
rq3_position = paper_text.index(r'\subsubsection{RQ3:')
detected_rq = 'RQ3' if rq3_position < m6_position else 'UNKNOWN'
assert detected_rq == 'RQ3'
assert rq_matches[detected_rq]
pd.DataFrame([{'metric': 'M6', 'detected_rq': detected_rq, 'rq_text': rq_matches[detected_rq], 'paper_sha256': sha256(paper_text.encode()).hexdigest()}])

In [ ]:
CONFIG = {
    'metric': 'M6',
    'expected_rq': 'RQ3',
    'unit_of_analysis': 'team_semester',
    'primary_outputs': ['planning_artifact_scope_t1', 'planning_message_content_t1', 'planning_evidence_concordance_t1'],
    'absence_policy': 'report_separately_not_as_score_floor',
    'llm_reprocessing_permitted': True,
    'inference': 'exploratory_descriptive',
}
assert CONFIG['expected_rq'] == detected_rq
assert CONFIG['absence_policy'] == 'report_separately_not_as_score_floor'
CONFIG

## 2. Auditoria do M6 legado

O M6 publicado reexpõe a coluna `t1_planning_score` do sinal legado. A nota é definida somente para equipes com texto de assunto de commit T1; ausência de commits deve continuar distinta de um score baixo.

In [ ]:
legacy_m6 = pd.read_csv(M6_PATH, dtype={'Semestre': str})
signals = pd.read_csv(SIGNALS_PATH, dtype={'Semestre': str})
assert legacy_m6.equals(signals[['ID_Equipe', 'Semestre', 't1_planning_score']])
assert legacy_m6.duplicated(['ID_Equipe', 'Semestre']).sum() == 0
legacy_audit = pd.DataFrame([
    ('team_semester_n', len(legacy_m6)),
    ('scored_n', int(legacy_m6.t1_planning_score.notna().sum())),
    ('missing_n', int(legacy_m6.t1_planning_score.isna().sum())),
    ('score_values', ', '.join(map(str, sorted(legacy_m6.t1_planning_score.dropna().unique())))),
    ('score_range', float(legacy_m6.t1_planning_score.max() - legacy_m6.t1_planning_score.min())),
], columns=['check', 'value'])
display(legacy_audit)
legacy_m6.sort_values(['Semestre', 'ID_Equipe'])

## 3. Evidência estrutural de planejamento

M6a reutiliza a regra determinística de artefatos de planejamento já calculada na Fase 2. Contagem de arquivos e delta de linhas representam escopo observável, não qualidade semântica. Equipes sem artefato recebem `0` no escopo, mas a ausência permanece identificada para M7.

In [ ]:
planning = pd.read_parquet(PLANNING_PATH).copy()
required = {'ID_Equipe', 'Semestre', 'pi_available', 'pi_file_count_t1', 'pi_line_delta_t1', 'planning_artifact_activity_t1'}
assert not (required - set(planning))
assert planning.duplicated(['ID_Equipe', 'Semestre']).sum() == 0
m6_structural = planning[['ID_Equipe', 'Semestre', 'pi_available', 'pi_file_count_t1', 'pi_line_delta_t1', 'planning_artifact_activity_t1']].copy()
m6_structural['planning_artifact_present_t1'] = m6_structural['pi_file_count_t1'].gt(0)
m6_structural['planning_scope_log1p_t1'] = np.log1p(m6_structural['pi_line_delta_t1'])
assert m6_structural.pi_available.all()
m6_structural.sort_values(['Semestre', 'ID_Equipe'])

## 4. Não redundância e decisão de reprocessamento

M6a (artefatos) e o score legado de M6 divergem empiricamente; portanto não são redundantes. Essa divergência também mostra que o score legado não pode validar qualidade arquitetural sozinho. M6b só deve ser gerado por reprocessamento LLM se o novo prompt exigir subtipos estruturados e evidência literal de cada decisão; caso contrário, deve ser marcado indisponível.

In [ ]:
comparison = legacy_m6.merge(m6_structural, on=['ID_Equipe', 'Semestre'], validate='one_to_one')
scored = comparison.dropna(subset=['t1_planning_score']).copy()
redundancy_tests = []
for structural_column in ['pi_file_count_t1', 'pi_line_delta_t1', 'planning_artifact_activity_t1']:
    rho, p_value = spearmanr(scored['t1_planning_score'], scored[structural_column])
    redundancy_tests.append({'legacy_m6': 't1_planning_score', 'structural_signal': structural_column, 'n': len(scored), 'spearman_rho': rho, 'p_value_exploratory': p_value})
redundancy_tests = pd.DataFrame(redundancy_tests)
assert len(scored) == 9
assert redundancy_tests['spearman_rho'].abs().max() < 0.5
display(redundancy_tests.round(3))
comparison.sort_values(['Semestre', 'ID_Equipe'])

## 5. Contrato para M6b, se reprocessado

O input permanece uma lista cronológica de assuntos T1, mas a resposta deve ser JSON estruturado: `planning_evidence_present`, `goals`, `architecture_or_design`, `task_decomposition`, `risk_or_dependency`, `evidence_quotes` e `insufficient_evidence`. Cada categoria deve usar apenas texto literal do input; ausência de evidência não pode ser inferida como baixa qualidade.

O reprocessamento deve registrar modelo, temperatura, hash do payload, versão do prompt, resposta bruta e evidências citadas. Sem esses campos, M6b não deve alimentar M9.

In [ ]:
m6_pipeline_decision = pd.DataFrame([
    ('M6a', 'adopt now', 'deterministic planning artifact presence and scope from planning_metrics.parquet'),
    ('M6b', 'regenerate conditionally', 'LLM structured extraction with literal evidence quotes from T1 subjects'),
    ('M6c', 'adopt after M6b', 'concordance matrix between structural and textual evidence; never a composite score'),
    ('M7', 'retain separately', 'absence of planning evidence is not an observed low-quality score'),
    ('M9', 'revise', 'associate M6a and M6b separately with outcomes; report missingness and no causal claim'),
], columns=['component', 'decision', 'reason'])
m6_pipeline_decision

In [ ]:
assert detected_rq == 'RQ3'
assert legacy_m6.equals(signals[['ID_Equipe', 'Semestre', 't1_planning_score']])
assert len(comparison) == 14
assert comparison['planning_artifact_present_t1'].eq(comparison['pi_file_count_t1'].gt(0)).all()
assert CONFIG['absence_policy'] == 'report_separately_not_as_score_floor'
assert CONFIG['inference'] == 'exploratory_descriptive'
m6_evidence_manifest = {
    'metric': 'M6', 'rq': detected_rq, 'unit_of_analysis': CONFIG['unit_of_analysis'],
    'legacy_source': str(SIGNALS_PATH.relative_to(PROJECT_ROOT)),
    'structural_source': str(PLANNING_PATH.relative_to(PROJECT_ROOT)),
    'legacy_scored_n': int(legacy_m6.t1_planning_score.notna().sum()),
    'legacy_missing_n': int(legacy_m6.t1_planning_score.isna().sum()),
    'llm_reprocessing_permitted': CONFIG['llm_reprocessing_permitted'],
}
m6_evidence_manifest